# Sales Email Agent Workflow

This notebook demonstrates how to build a **multi-agent AI sales email generation pipeline** using the OpenAI Agents SDK and a local Ollama model.

## Objectives
- Configure the environment
- Send emails through SMTP
- Create multiple sales agents with different writing styles
- Generate emails in parallel
- Expose email sending as a tool
- Use a decision agent to select the best email
- Automatically send the selected email

## Step 1 – Import Libraries

Import all required packages for AI agents, SMTP email delivery, asynchronous execution, tracing, and environment configuration.

In [1]:
from dotenv import load_dotenv
import requests
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, ModelSettings
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from openai.types.responses import ResponseTextDeltaEvent
from agents.extensions.visualization import draw_graph
from openai.types.responses import ResponseTextDeltaEvent
import os
import html
import asyncio
import smtplib
from email.message import EmailMessage
load_dotenv(override=True)

MODEL_NAME = "llama3.2:latest"
ollama_client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # Dummy value required by the client
)

ollama_model = OpenAIChatCompletionsModel(
    model=MODEL_NAME,
    openai_client=ollama_client,
)

## Step 2 – Load Environment Variables

Read SMTP credentials and configuration from the `.env` file so sensitive information is not hardcoded.

In [2]:
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

if EMAIL_ADDRESS:
    print("Email address is set")
else:
    print("Email address is not set")

if EMAIL_SMTP_SERVER:
    print("SMTP server is set")
else:
    print("SMTP server is not set")

if EMAIL_APP_PASSWORD:
    print("App password is set")
else:
    print("App password is not set")

USE_EMAIL = EMAIL_ADDRESS and EMAIL_SMTP_SERVER and EMAIL_APP_PASSWORD

if USE_EMAIL:
    print("Email is set up and we will try using it")
else:
    print("Email is not set up; we will send push notifications instead")

Email address is set
SMTP server is set
App password is set
Email is set up and we will try using it


## Step 3 – Create Email Sending Function

Implement a reusable SMTP function that can send both plain-text and HTML emails.

In [3]:
def send_email(subject, text_body, html_body):
    if not EMAIL_ADDRESS or not EMAIL_SMTP_SERVER or not EMAIL_APP_PASSWORD:
        raise ValueError("Email configuration is missing from .env")

    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    try:
        with smtplib.SMTP(EMAIL_SMTP_SERVER, 587, timeout=30) as server:
            server.ehlo()
            server.starttls()
            server.ehlo()
            server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
            server.send_message(msg)

        print("Email sent successfully")

    except Exception as error:
        print("Email sending failed:", repr(error))
        raise

## Step 4 – Test SMTP Connection

Verify that the email configuration works correctly before integrating it with AI agents.

In [4]:
send_email("Testing testing 123", "Fingers crossed..", "<html><body><strong>Fingers</strong> crossed..</body></html>")

Email sent successfully


## Step 5 – Define Agent Prompts

Create prompts describing the company and define different sales-writing personalities.

In [5]:
intro = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write emails.
"""

instructions1 = intro + "Your email style is professional, serious, with gravitas and credibility."
instructions2 = intro + "Your email style is witty, engaging, and humorous."
instructions3 = intro + "Your email style is concise, to the point, in the style of a busy senior executive."

## Step 6 – Create Sales Agents

Instantiate multiple AI agents, each producing a different email style.

In [6]:
sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model=ollama_model)
sales_agent2 = Agent(name="Humorous Sales Agent", instructions=instructions2, model=ollama_model)
sales_agent3 = Agent(name="Executive Sales Agent", instructions=instructions3, model=ollama_model)

## Step 7 – Generate a Sample Email

Run one sales agent individually to observe streamed output.

In [ ]:


result = Runner.run_streamed(
    sales_agent1,
    input="Write a cold sales email",
)

async for event in result.stream_events():
    if (
        event.type == "raw_response_event"
        and isinstance(event.data, ResponseTextDeltaEvent)
    ):
        print(event.data.delta, end="", flush=True)

OPENAI_API_KEY is not set, skipping trace export


Subject: Expert Guidance for Enhanced SOC2 Compliance

Dear [Recipient's Name],

As a respected [title] at [Company Name], I'm sure you understand the importance of maintaining the highest standards of cybersecurity and governance in today's ever-evolving regulatory landscape.

At ComplAI, we've been helping forward-thinking organizations like yours navigate the complexities of SOC2 compliance and audit preparation. Our innovative SaaS tool, powered by AI, streamlines the process, ensuring you're always up-to-date and compliant.

Our solution provides real-time monitoring, automatic reporting, and robust analytics to identify potential risks and vulnerabilities. With ComplAI, you can:

- Enhance your risk management strategy with data-driven insights
- Achieve faster audit cycles and reduced costs
- Demonstrate increased confidence in your organizational security posture

I'd love to schedule a brief call to discuss how ComplAI can specifically support your compliance and audit prepara

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is no

## Step 8 – Parallel Email Generation

Execute all sales agents concurrently to produce multiple email candidates.

In [8]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

Subject: Unlock Enhanced SOC2 Compliance with AI-Powered Expertise

Dear [Recipient's Name],

I encourage you to take a moment to review your organization's compliance posture. As a responsible business leader, you understand the importance of maintaining the highest standards of security, availability, and governance. However, navigating the complexities of SOC2 compliance can be a daunting task, especially for mid-sized to large enterprises.

At ComplAI, we've developed a cutting-edge SaaS solution that empower organizations like yours to excel inSOC2 audits. Our intuitive tool leverages AI technology to streamline the entire compliance lifecycle, from risk assessment to reporting. By partnering with us, you can ensure peace of mind and a reduced risk profile, while also meeting the evolving regulatory requirements.

With ComplAI's innovative approach, you can expect:

- AI-driven risk assessments to minimize audit coverage
- Automated reporting and assessment to reduce manual effort

## Step 9 – Convert Email Sending into a Tool

Wrap the email function using `@function_tool` so AI agents can invoke it safely.

In [9]:
import html

@function_tool
def send_email_tool(
    subject: str,
    text_body: str,
    html_body: str = "",
) -> str:
    """Send the selected sales email."""

    print("✅ TOOL EXECUTED")
    print("Subject:", subject)
    print("Text body:", text_body)
    print("HTML body:", repr(html_body))

    # Replace missing or malformed HTML with safe HTML generated
    # from the plain-text email.
    if not html_body or "<" not in html_body or ">" not in html_body:
        escaped_body = html.escape(text_body)
        html_body = (
            "<html><body>"
            f"<p>{escaped_body.replace(chr(10), '<br>')}</p>"
            "</body></html>"
        )

    send_email(subject, text_body, html_body)
    return "Email sent successfully"

## Step 10 – Inspect Tool Schema

Review the automatically generated JSON schema exposed to the language model.

In [10]:

send_email_tool.params_json_schema

{'properties': {'subject': {'title': 'Subject', 'type': 'string'},
  'text_body': {'title': 'Text Body', 'type': 'string'},
  'html_body': {'default': '', 'title': 'Html Body', 'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

## Step 11 – Build Decision Agent

Create a supervising agent that evaluates all generated emails and must call the email tool exactly once.

In [16]:
decision = """
Choose the best cold sales email from the provided options.
Imagine you are the customer and select the email you would be most likely to respond to.

Call send_email_tool exactly once.

Use exactly these arguments:
- subject: only the subject line
- text_body: only the complete plain-text email body

Do not include html_body.
Do not include the word "Subject:" inside text_body.
Do not add any other parameters.
After the tool completes, reply only: Email sent successfully.
"""

sales_sender = Agent(
    name="Sales Sender",
    instructions=decision,
    model=ollama_model,
    tools=[send_email_tool],
    model_settings=ModelSettings(
        tool_choice="required",
        parallel_tool_calls=False,
    ),
)

## Step 12 – End-to-End Workflow

Generate multiple email candidates, allow the decision agent to choose the strongest version, invoke the tool, and send the email automatically.

In [18]:
with trace("Sales selection workflow with sending"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

    outputs = [result.final_output for result in results]

    emails = "\n\n--- EMAIL OPTION ---\n\n".join(outputs)

    response = await Runner.run(
        sales_sender,
        emails,
    )

print("Final response:", response.final_output)



✅ TOOL EXECUTED
Subject: Boost Your Organization
Text body: As a [Recipient
HTML body: ''
Email sent successfully
Final response: Based on the provided options, I would recommend the first email as the best cold sales email. Here's why:

* The subject line is attention-grabbing and relevant to the recipient's role and responsibilities.
* The text body is well-written and informative, providing a clear overview of the benefits of using ComplAI's SaaS tool. The email effectively highlights the importance of SOC2 compliance and how ComplAI can help organizations navigate audits with confidence.
* The email includes several compelling benefits, such as reducing compliance cycles by up to 50% and saving an average of $100,000 per year on compliance costs.
* The call-to-action (CTA) is clear and inviting, suggesting a complimentary 30-minute call to discuss the recipient's compliance strategy and how ComplAI can support it.

Overall, this email is well-structured, engaging, and effectively c

## Summary

In this notebook you learned how to:

- Build multiple specialized AI agents
- Execute agents concurrently
- Expose Python functions as agent tools
- Enforce tool usage with `tool_choice="required"`
- Build a supervisor agent for decision making
- Automate email delivery using SMTP

This architecture demonstrates an effective multi-agent workflow for autonomous content generation and tool execution.

## Summary

In this notebook you learned how to:

- Build multiple specialized AI agents
- Execute agents concurrently
- Expose Python functions as agent tools
- Enforce tool usage with `tool_choice="required"`
- Build a supervisor agent for decision making
- Automate email delivery using SMTP

This architecture demonstrates an effective multi-agent workflow for autonomous content generation and tool execution.